# Chat Completions 기초

건강·피트니스 상담 애플리케이션을 만든다고 해봅시다. 가장 먼저 필요한 것은 모델과 말을 주고받는 통로입니다. 이 노트북에서는 **Microsoft Foundry** 프로젝트에 연결해 **`openai`** SDK로 질문을 보내고 답을 받아오는 흐름을 직접 만들어 봅니다. `azure-ai-projects`의 `AIProjectClient`로 프로젝트에 연결하고 `get_openai_client()`로 OpenAI 호환 클라이언트를 얻습니다.

**🎯 미션**

1. Foundry 프로젝트에 연결된 OpenAI 호환 클라이언트를 초기화합니다.
2. 피트니스 어시스턴트 역할을 하는 프롬프트 템플릿을 만들고 건강 질문에 답하게 합니다.
3. 최신 표준 인터페이스인 **Responses API**로 같은 요청을 보내 봅니다.

> **사전 준비**: `az login`과 [03. 인증 구성](../03-authentication/README.md)이 완료되어 있어야 합니다.

> 이전 버전에서 사용하던 `azure-ai-inference` 패키지(ChatCompletionsClient)는 2026-05-30에 지원이 종료되어 `openai` 패키지로 교체되었습니다.

## 🏋️ 건강-피트니스 관련 안내
> **이 예제는 데모 목적일 뿐이며 실제 의학적 조언을 제공하지 않습니다.**


## 1. 초기 설정

환경 변수를 로드하고 프로젝트에 연결합니다. 실행 전 `az login`과 [03. 인증 구성](../03-authentication/README.md)이 완료되어 있어야 합니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# 상위 폴더의 .env 로드
load_dotenv(Path().absolute().parent / ".env")

endpoint = os.environ["PROJECT_ENDPOINT"]
chat_model = os.environ["MODEL_NAME"]                    # 배포 이름 (예: gpt-5-mini)
embedding_model = os.environ["TEXT_EMBEDDING_MODEL"]     # 배포 이름 (예: text-embedding-3-small)

# Entra ID(keyless) 인증 — 사전에 `az login` 필요
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())

# 프로젝트에 연결된 OpenAI 호환 클라이언트 (chat/responses 모두 이 클라이언트 사용)
client = project.get_openai_client()
print("✅ AIProjectClient / OpenAI client 준비 완료")


### 프롬프트 템플릿

친절하고 안내 문구를 제공하는 피트니스 어시스턴트 역할의 **system** 메시지를 정의하고, 사용자 입력을 **user** 메시지로 전달합니다.

In [ ]:
def chat_with_fitness_assistant(user_input: str) -> str:
    """시스템 프롬프트와 함께 chat completions를 실행합니다."""
    system_text = (
        "You are FitChat GPT, a friendly fitness assistant.\n"
        "Always remind users: I'm not a medical professional.\n"
        "Answer with empathy and disclaimers."
    )

    response = client.chat.completions.create(
        model=chat_model,  # Foundry의 모델 "배포 이름"
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_input},
        ],
    )
    return response.choices[0].message.content

print("Defined a helper function to do chat completions.")

## 2. Chat Completions 실행해보기

건강이나 피트니스에 관한 사용자의 질문으로 함수를 호출하고 결과를 확인합니다. 질문을 자유롭게 수정해보세요!

In [ ]:
user_question = "How can I start a beginner workout routine at home?"
reply = chat_with_fitness_assistant(user_question)
print("🗣️ User:", user_question)
print("🤖 Assistant:", reply)

## 3. 채우기 형식의 프롬프트 템플릿

시스템 메시지에 **userName**, **goal** 같은 플레이스홀더를 넣어 확장할 수 있습니다.

In [ ]:
def chat_with_template(user_input: str, user_name: str, goal: str) -> str:
    system_template = (
        "You are FitChat GPT, an AI personal trainer for {name}.\n"
        "Your user wants to achieve: {goal}.\n"
        "Remind them you're not a medical professional. Offer friendly advice."
    )
    system_prompt = system_template.format(name=user_name, goal=goal)

    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input},
        ],
    )
    return response.choices[0].message.content

templated_user_input = "What kind of home exercise do you recommend for a busy schedule?"
assistant_reply = chat_with_template(
    templated_user_input,
    user_name="Jordan",
    goal="increase muscle tone and endurance",
)
print("🗣️ User:", templated_user_input)
print("🤖 Assistant:", assistant_reply)

## 4. (보너스) Responses API 맛보기

Foundry의 최신 표준 인터페이스는 **Responses API**입니다. 상태 유지(stateful) 대화·에이전트가 이 API 위에서 동작하며, [05. Agent Service](../05-agent-service/README.md)에서 본격적으로 사용합니다.

In [ ]:
response = client.responses.create(
    model=chat_model,
    input="Give me one quick stretching tip for office workers.",
)
print("🤖", response.output_text)

## ✅ 미션 완료

**무엇을 만들었나:**

- ✓ Foundry 프로젝트에 연결된 OpenAI 호환 클라이언트
- ✓ 피트니스 어시스턴트 프롬프트 템플릿 (플레이스홀더를 채우는 확장 버전 포함)
- ✓ Responses API로 보낸 첫 요청

`openai` 클라이언트 하나로 chat completions와 responses를 모두 사용했습니다. 다음 노트북에서는 같은 클라이언트로 **임베딩**을 생성합니다 → [02-embeddings.ipynb](02-embeddings.ipynb)
